# WSMTE — 01_data_prep.ipynb
**LOCAL PC** | Steps 2–8 from DATA_PIPELINE.md  
Loads finbert outputs + price data, aggregates to daily, merges by trading date.

**Prerequisites**: Run `notebooks/03_finbert_inference.ipynb` on Kaggle GPU first, then
download the 3 output CSVs into `data/finbert_outputs/`.

In [1]:
import sys, os

# Find project root by walking up until config/config.py is found
_search = os.path.abspath('')
while not os.path.exists(os.path.join(_search, 'config', 'config.py')):
    _parent = os.path.dirname(_search)
    if _parent == _search:
        raise RuntimeError("Cannot find project root — is config/config.py missing?")
    _search = _parent

PROJECT_ROOT = _search
sys.path.insert(0, PROJECT_ROOT)

import pandas as pd
from config.config import CONFIG

# Make every CONFIG path absolute so no cell depends on CWD
_path_keys = [k for k in CONFIG if isinstance(CONFIG[k], str) and
              (k.endswith('_dir') or k.endswith('_file') or k == 'ablation_results')]
for k in _path_keys:
    CONFIG[k] = os.path.join(PROJECT_ROOT, CONFIG[k])

print(f"Project root : {PROJECT_ROOT}")
print(f"raw_data_dir : {CONFIG['raw_data_dir']}")
print(f"nifty50_ohlcv.csv exists: {os.path.exists(os.path.join(CONFIG['raw_data_dir'], 'nifty50_ohlcv.csv'))}")

Project root : d:\WSMTE
raw_data_dir : d:\WSMTE\data/raw/
nifty50_ohlcv.csv exists: True


## Step 2 — Load Raw Datasets

In [2]:
# ── Price data ──────────────────────────────────────────────────────────────
price_df = pd.read_csv(CONFIG['raw_data_dir'] + 'nifty50_ohlcv.csv')
price_df['date'] = pd.to_datetime(price_df['Date']).dt.date
price_df = price_df.sort_values('date').reset_index(drop=True)
print(f"Price data: {price_df.shape} | {price_df['date'].min()} to {price_df['date'].max()}")

Price data: (1092, 7) | 2020-01-01 to 2024-05-31


In [3]:
# ── Kotekar dataset (company-level) ─────────────────────────────────────────
# Columns: datePublished, company, symbol, headline, description,
#          articleBody, tags, author, url
kotekar = pd.read_csv(CONFIG['raw_data_dir'] + 'kotekar_news.csv')
kotekar['date'] = pd.to_datetime(kotekar[CONFIG['kotekar_date_col']]).dt.date
kotekar = kotekar[
    (kotekar['date'] >= pd.to_datetime(CONFIG['start_date']).date()) &
    (kotekar['date'] <= pd.to_datetime(CONFIG['end_date']).date())
]
print(f"Kotekar: {kotekar.shape}")

Kotekar: (2719, 10)


In [4]:
# ── Kaggle Dataset 1 (Jan 2020 – Apr 15, 2021, market-level) ────────────────────
# Columns: Date (DD/MM/YY), Title, URL, sentiment, confidence
# NOTE: existing sentiment/confidence ignored — FinBERT reruns for consistency
kaggle1 = pd.read_csv(CONFIG['raw_data_dir'] + 'kaggle_news_1.csv')
kaggle1['date'] = pd.to_datetime(kaggle1[CONFIG['kaggle1_date_col']], format='%d/%m/%y').dt.date
kaggle1 = kaggle1[
    (kaggle1['date'] >= pd.to_datetime(CONFIG['start_date']).date()) &
    (kaggle1['date'] <= pd.to_datetime('2021-04-15').date())
]
print(f"Kaggle1: {kaggle1.shape} | {kaggle1['date'].min()} to {kaggle1['date'].max()}")

Kaggle1: (57931, 6) | 2020-01-01 to 2021-04-15


In [5]:
# ── Kaggle Dataset 2 (Jan 2022 – Apr 2024, market-level) ────────────────────
# Columns: Archive, Date (DD-MM-YYYY), Headline, Headline link
kaggle2 = pd.read_csv(CONFIG['raw_data_dir'] + 'kaggle_news_2.csv')
kaggle2['date'] = pd.to_datetime(kaggle2[CONFIG['kaggle2_date_col']], format='%d-%m-%Y').dt.date
kaggle2 = kaggle2[
    (kaggle2['date'] >= pd.to_datetime('2022-01-01').date()) &
    (kaggle2['date'] <= pd.to_datetime(CONFIG['end_date']).date())
]
print(f"Kaggle2: {kaggle2.shape} | {kaggle2['date'].min()} to {kaggle2['date'].max()}")

Kaggle2: (275184, 5) | 2022-01-01 to 2024-05-29


## Steps 3–6 — Load FinBERT / mDeBERTa Outputs
(Generated by `notebooks/03_finbert_inference.ipynb` on Kaggle GPU)

In [6]:
# ── kotekar_sentiment.csv ───────────────────────────────────────────────────
# Cols: date, company, symbol, polarity_company, subjectivity
kotekar_sent = pd.read_csv(CONFIG['kotekar_sentiment_file'])
kotekar_sent['date'] = pd.to_datetime(kotekar_sent['date']).dt.date
print(f"Kotekar sentiment: {kotekar_sent.shape}")
print(f"  polarity_company range: {kotekar_sent['polarity_company'].min():.3f} to {kotekar_sent['polarity_company'].max():.3f}")
print(f"  subjectivity range:     {kotekar_sent['subjectivity'].min():.3f} to {kotekar_sent['subjectivity'].max():.3f}")

Kotekar sentiment: (2719, 5)
  polarity_company range: -0.970 to 0.946
  subjectivity range:     0.006 to 0.998


In [7]:
# ── kaggle1_polarity.csv ────────────────────────────────────────────────────
# Cols: date, polarity_market
kaggle1_pol = pd.read_csv(CONFIG['kaggle1_polarity_file'])
kaggle1_pol['date'] = pd.to_datetime(kaggle1_pol['date']).dt.date
print(f"Kaggle1 polarity: {kaggle1_pol.shape}")

# ── kaggle2_polarity.csv ────────────────────────────────────────────────────
kaggle2_pol = pd.read_csv(CONFIG['kaggle2_polarity_file'])
kaggle2_pol['date'] = pd.to_datetime(kaggle2_pol['date']).dt.date
print(f"Kaggle2 polarity: {kaggle2_pol.shape}")

Kaggle1 polarity: (57931, 2)
Kaggle2 polarity: (263127, 2)


## Step 7 — Aggregate All Sentiment to Daily Level

In [8]:
# ── Company-level: mean per trading day ─────────────────────────────────────
company_daily = kotekar_sent.groupby('date').agg(
    polarity_company=('polarity_company', 'mean'),
    polarity_company_max=('polarity_company', lambda x: x.loc[x.abs().idxmax()]),
    subjectivity=('subjectivity', 'mean')
).reset_index()
print(f"Company daily: {company_daily.shape}")

# ── Market-level: combine Kaggle 1 + 2, no date overlap ─────────────────────
# Kaggle1 covers up to Apr 15, 2021, Kaggle2 from Jan 2022; gap May–Dec 2021
market_combined = pd.concat([
    kaggle1_pol[['date', 'polarity_market']],
    kaggle2_pol[['date', 'polarity_market']]
]).sort_values('date').reset_index(drop=True)

market_daily = market_combined.groupby('date').agg(
    polarity_market=('polarity_market', 'mean'),
    polarity_market_max=('polarity_market', lambda x: x.loc[x.abs().idxmax()]),
).reset_index()
print(f"Market daily: {market_daily.shape}")

Company daily: (752, 4)
Market daily: (1260, 3)


## Step 8 — Merge All Sources by Trading Date

In [9]:
# Left join on price data (trading days are the anchor)
df = price_df.merge(company_daily, on='date', how='left')
df = df.merge(market_daily, on='date', how='left')

# Fill gap period May–Dec 2021 + any other missing
# polarity_market = 0 (neutral), polarity_company = 0, subjectivity = 0.5
df['polarity_market']  = df['polarity_market'].fillna(CONFIG['missing_polarity'])
df['polarity_company'] = df['polarity_company'].fillna(CONFIG['missing_polarity'])
df['subjectivity']     = df['subjectivity'].fillna(CONFIG['missing_subjectivity'])
df['polarity_market_max']  = df['polarity_market_max'].fillna(CONFIG['missing_polarity'])
df['polarity_company_max'] = df['polarity_company_max'].fillna(CONFIG['missing_polarity'])


df = df.sort_values('date').reset_index(drop=True)
print(f"Merged shape: {df.shape}")
print(f"Date range:   {df['date'].min()} to {df['date'].max()}")
print(f"Missing values:{df[['polarity_market','polarity_company','subjectivity']].isnull().sum()}")

Merged shape: (1092, 12)
Date range:   2020-01-01 to 2024-05-31
Missing values:polarity_market     0
polarity_company    0
subjectivity        0
dtype: int64


In [10]:
# Verify gap period May–Dec 2021 has polarity_market = 0
gap_start = pd.to_datetime(CONFIG['gap_start']).date()
gap_end   = pd.to_datetime(CONFIG['gap_end']).date()
gap = df[(df['date'] >= gap_start) & (df['date'] <= gap_end)]
assert (gap['polarity_market'] == 0.0).all(), "Gap period fill failed!"
print(f"Gap period ({gap_start} to {gap_end}): {len(gap)} trading days, polarity_market = 0 ✓")

Gap period (2021-04-16 to 2021-12-31): 178 trading days, polarity_market = 0 ✓


In [11]:
df.to_csv(CONFIG['processed_data_dir'] + 'merged_data.csv', index=False)
print("Saved → data/processed/merged_data.csv")

Saved → data/processed/merged_data.csv
